In [1]:
# import Pkg; Pkg.add(["NCDatasets", "Interpolations"])
#
# Land Mask file used from:
# Mikelsons, Karlis; Wang, Menghua; Jiang, Lide; Wang, Xiao-Long (2021), 
# “Global land mask for satellite ocean color remote sensing”, 
# Mendeley Data, V1, doi: 10.17632/9r93m9s7cw.1

using NCDatasets

"""
    apply_noaa_watermask!(mask, xi, yi; path="watermask.nc", verbose=true)

Specialized for NOAA STAR watermask:
  - variables: watermask (Int8, 1=water, 0=land/ice), lon (1-D), lat (1-D)
  - dimensions order: (lon, lat)

Reads only the needed lon/lat window, snaps (xi, yi) to nearest grid cell
with direct index math, and AND-merges into `mask` in place.

Assumptions:
  - `xi`, `yi` are 1-D axes from DIVAnd_rectdom (lon-like, lat-like).
  - `mask` is Bool with size (length(xi), length(yi)).
"""
function apply_noaa_watermask!(mask::AbstractMatrix{Bool},
                               xi::AbstractVector{<:Real},
                               yi::AbstractVector{<:Real};
                               path::AbstractString="watermask.nc",
                               verbose::Bool=true)

    (size(mask,1), size(mask,2)) == (length(xi), length(yi)) ||
        error("mask must be (length(xi), length(yi)); got mask=$(size(mask)) axes=($(length(xi)),$(length(yi))).")

    xi_min, xi_max = extrema(xi)
    yi_min, yi_max = extrema(yi)

    ds = NCDataset(path, "r")
    try
        lon = vec(Array(ds["lon"]))   # expected regular: step ≈ 1/480°
        lat = vec(Array(ds["lat"]))

        # Handle descending axes in file
        lon_rev = length(lon) > 1 && lon[2] < lon[1]
        lat_rev = length(lat) > 1 && lat[2] < lat[1]
        lon_sorted = lon_rev ? reverse(lon) : lon
        lat_sorted = lat_rev ? reverse(lat) : lat

        # Step sizes (assumed constant)
        dlon = length(lon_sorted) > 1 ? lon_sorted[2] - lon_sorted[1] : 1.0
        dlat = length(lat_sorted) > 1 ? lat_sorted[2] - lat_sorted[1] : 1.0

        # Pad by one cell
        i1s = clamp(searchsortedfirst(lon_sorted, xi_min - dlon), 1, length(lon_sorted))
        i2s = clamp(searchsortedlast( lon_sorted, xi_max + dlon),  1, length(lon_sorted))
        j1s = clamp(searchsortedfirst(lat_sorted, yi_min - dlat), 1, length(lat_sorted))
        j2s = clamp(searchsortedlast( lat_sorted, yi_max + dlat),  1, length(lat_sorted))

        # Map to file indices if reversed
        if lon_rev
            Nlon = length(lon_sorted)
            i1, i2 = Nlon - i2s + 1, Nlon - i1s + 1
        else
            i1, i2 = i1s, i2s
        end
        if lat_rev
            Nlat = length(lat_sorted)
            j1, j2 = Nlat - j2s + 1, Nlat - j1s + 1
        else
            j1, j2 = j1s, j2s
        end

        verbose && println("→ Reading subset: lon[$i1:$i2] × lat[$j1:$j2]")

        wm_sub = Array(ds["watermask"][i1:i2, j1:j2])  # Int8, no casting
        lon_sub_file = lon[i1:i2]
        lat_sub_file = lat[j1:j2]

        # Re-orient subset to ascending axes for simple index math
        if lon_rev
            wm_sub = reverse(wm_sub, dims=1)
            lon_sub = reverse(lon_sub_file)
        else
            lon_sub = lon_sub_file
        end
        if lat_rev
            wm_sub = reverse(wm_sub, dims=2)
            lat_sub = reverse(lat_sub_file)
        else
            lat_sub = lat_sub_file
        end

        # Nearest-neighbor indices on regular grids (no large temp arrays)
        # idx = round( (x - x0)/dx ) + 1, clamped to [1, N]
        @inline nearest_idx(x, x0, dx, n) = clamp(round(Int, (x - x0)/dx) + 1, 1, n)

        nx, ny = length(xi), length(yi)
        ix = Vector{Int}(undef, nx)
        @inbounds for k in 1:nx
            ix[k] = nearest_idx(xi[k], lon_sub[1], dlon, length(lon_sub))
        end
        iy = Vector{Int}(undef, ny)
        @inbounds for k in 1:ny
            iy[k] = nearest_idx(yi[k], lat_sub[1], dlat, length(lat_sub))
        end

        # Gather the subset mask on your grid: this allocates an Int8 matrix of size(nx, ny)
        sel = wm_sub[ix, iy]          # Int8 values 0/1
        mask_from_file = sel .>= 1    # Bool: true where water

        # AND-merge: keep only cells that are already valid AND water
        mask .= mask .& mask_from_file
        verbose && println("✓ Watermask applied. Kept $(count(mask)) of $(length(mask)) cells.")
        return mask
    finally
        close(ds)
    end
end



apply_noaa_watermask!

In [2]:
# Description:
# This script reads the first band (surface layer) of a multi-band GeoTIFF file.
# It identifies pixels with missing data and uses DIVAnd.jl to perform a 2D
# spatial interpolation to fill these gaps. The final output raster perfectly preserves
# the original data and only fills in the missing values.
#
# Instructions:
# 1. Make sure you have the required packages:
#    using Pkg
#    Pkg.add(["DIVAnd", "Rasters", "GDAL", "Statistics", "ArchGDAL"])
# 2. Update the `input_geotiff_path` and `output_geotiff_path` variables below.
# 3. Adjust `len` (correlation length) and `epsilon2` (error variance) as needed for your specific dataset.

using DIVAnd         # Package for Data-Interpolating Variational Analysis
using Rasters        # Package for reading and manipulating raster data
using GDAL           # GDAL backend for Rasters.jl to support GeoTIFFs
import ArchGDAL      # Explicitly import ArchGDAL to register the backend for Rasters.jl
using Statistics     # For calculating mean for logging

# --- 1. DEFINE FILE PATHS AND PARAMETERS ---
# input is ~1deg ~= 100km per pixel
const lat_lon_len = 10  # in units of pixels

# --- Interpolation Parameters ---
# Correlation length in (x, y) directions.
# These values are crucial for good results and depend on your data's spatial characteristics.
# They are in the units of your data's coordinate system.
const len_x = lat_lon_len  
const len_y = lat_lon_len
const len = (len_x, len_y) # 2D length tuple

# Epsilon2: normalized variance of the observation error. A small value assumes
# the existing data points are accurate.
# This prevents the result from being smoothed into the mean and preserves 
# data variance. A value of 1 allows for significant smoothing.
const epsilon2 = 10

# --- 2. LOAD DATA AND EXTRACT OBSERVATIONS ---

const input_fname = "woa23_all_p00_01.nc_p_sd.tif"
const input_geotiff_path = "data/raw/$(input_fname)"
# Modified output path to indicate a 2D analysis
const output_geotiff_path = "data/filled/$(input_fname).filled.2D.len$(lat_lon_len).e$(epsilon2).tif"

# --- Check if input file exists ---
if !isfile(input_geotiff_path)
    error("""
    Input file not found at: $(input_geotiff_path)
    Please create a dummy GeoTIFF for testing or update the path.
    You can create a test file with GDAL or a GIS software like QGIS.
    """)
end

@info "Loading GeoTIFF from: $(input_geotiff_path) (Band 1 only)"
# Load the entire raster first.
full_raster = Raster(input_geotiff_path)
# Then, select the first band to create a 2D surface raster.
surface_raster = full_raster[Band(1)]


# Add a check to ensure the loaded raster is 2-dimensional.
if ndims(surface_raster) != 2
    error("Input raster is not 2-dimensional as expected. It has $(ndims(surface_raster)) dimensions. Please check the input file and that only one band was loaded.")
end

# Find the indices of all valid (not missing) data points from the surface raster.
valid_indices = findall(!ismissing, surface_raster)
@info "Found $(length(valid_indices)) valid data points out of a total of $(length(surface_raster))."

# Pre-allocate arrays for observation coordinates (x, y) and values (f)
nobs = length(valid_indices)
x_obs = Vector{Float64}(undef, nobs)
y_obs = Vector{Float64}(undef, nobs)
f_obs = Vector{Float64}(undef, nobs) # The actual pixel value

# Extract the coordinates and values for each valid point
x_coords = dims(surface_raster, X)
y_coords = dims(surface_raster, Y)

for (i, idx) in enumerate(valid_indices)
    # Get the (lon, lat) index for the i-th valid pixel
    ix, iy = Tuple(idx)

    x_obs[i] = x_coords[ix]
    y_obs[i] = y_coords[iy]
    f_obs[i] = surface_raster[idx]
end

@info "Observations extracted. Mean observed value: $(mean(f_obs))"

# --- 3. DEFINE THE INTERPOLATION GRID AND ANOMALIES ---

# Calculate the mean of all valid observations. This will serve as our background field.
const mean_f_obs = mean(f_obs)
@info "Calculated background field mean value: $(mean_f_obs)"

# NEW: Calculate the observation anomalies. We will interpolate these values
# instead of the absolute values. This prevents the analysis from collapsing
# to the mean field.
const f_anomalies = f_obs .- mean_f_obs
@info "Calculated observation anomalies. Mean anomaly should be near zero: $(mean(f_anomalies))"

# The output grid will match the input raster's dimensions and coordinates exactly.
# Define the output grid using the dimensions of the surface raster.
# Store the original coordinate order before sorting.
original_x_coords = Float64.(val(dims(surface_raster, X)))
original_y_coords = Float64.(val(dims(surface_raster, Y)))

# CRITICAL FIX: Ensure coordinate vectors are sorted in ascending order to prevent DomainError in DIVAnd.
grid_x = sort(original_x_coords)
grid_y = sort(original_y_coords)

@info "lenx:$(length(grid_x)) ; leny:$(length(grid_y))"

# DIVAnd_rectdom creates the domain, grid metrics, and coordinate arrays for the 2D analysis.
mask, (pm, pn), (xi, yi) = DIVAnd_rectdom(grid_x, grid_y)

# Keep WATER pixels (assumes watermask.nc has 1=water)
xi_1d = vec(xi[:, 1])
yi_1d = vec(yi[1, :])
apply_noaa_watermask!(mask, xi_1d, yi_1d; path="watermask.nc")




@info "Interpolation grid created with size $(size(mask))."

# --- 4. RUN THE INTERPOLATION ON THE ANOMALIES ---

@info "Starting 2D DIVAnd interpolation on anomalies... (This may take some time)"
# We interpolate the ANOMALIES. The background field for anomalies is implicitly zero.
@time fi_anomalies, s = DIVAndrun(
    mask,
    (pm, pn),
    (xi, yi),
    (x_obs, y_obs),
    f_anomalies, # Use the anomalies as the observed values
    len,
    epsilon2;
    # Use an iterative solver for large problems to avoid out-of-memory errors.
    inversion = :solve,
    tol = 1e-5,       # Tolerance for the solver to converge
    maxit = 2000      # Maximum number of iterations
)
@info "Interpolation of anomalies complete."

# Reconstruct the full interpolated field by adding the background mean back.
const fi_interpolated = fi_anomalies .+ mean_f_obs
@info "Reconstructed the full interpolated field."

# --- 5. SAVE THE INTERPOLATED FIELD ---

@info "Saving the interpolated field..."
# FIX: Re-orient the interpolated field to match the original raster's orientation.
fi_reoriented = fi_interpolated
# Check if the original Y-axis was descending (north-up).
if original_y_coords[1] > original_y_coords[end]
    @info "Original Y-axis was descending (north-up). Re-orienting interpolated data."
    # The interpolation was done on a south-up grid. We flip it vertically to match.
    # The Y dimension corresponds to the 2nd dimension of the matrix.
    fi_reoriented = reverse(fi_interpolated, dims=2)
end

# The final data is the reoriented, fully interpolated field.
final_data = fi_reoriented

@info "Saving filled raster to: $(output_geotiff_path)"
# Extract the Coordinate Reference System (CRS) from the original raster.
const input_crs = Rasters.crs(surface_raster)

# Create a new 2D Raster object.
# We must build new dimensions from our ORIGINAL grid vectors to ensure the
# output raster is correctly oriented.
#
# CRITICAL FIX: The GeoTIFF format requires a perfectly regular grid.
# We use `LinRange` to generate a perfectly regular
# coordinate sequence, ensuring GDAL can write the file.
new_dims = (
    X(LinRange(original_x_coords[1], original_x_coords[end], length(original_x_coords)); crs=input_crs, dim=X),
    Y(LinRange(original_y_coords[1], original_y_coords[end], length(original_y_coords)); crs=input_crs, dim=Y)
)

filled_raster = Raster(final_data, new_dims; missingval=missingval(surface_raster))

# Write the new raster to a GeoTIFF file
write(output_geotiff_path, filled_raster; force=true)

@info "Processing finished successfully."



[ Info: Loading GeoTIFF from: data/raw/woa23_all_p00_01.nc_p_sd.tif (Band 1 only)
[ Info: Found 19415 valid data points out of a total of 64800.
[ Info: Observations extracted. Mean observed value: 0.08497965024806364
[ Info: Calculated background field mean value: 0.08497965024806364
[ Info: Calculated observation anomalies. Mean anomaly should be near zero: 4.098933113836272e-17
[ Info: lenx:360 ; leny:180


→ Reading subset: lon[1:172321] × lat[480:86400]
✓ Watermask applied. Kept 42810 of 64800 cells.


[ Info: Interpolation grid created with size (360, 180).
[ Info: Starting 2D DIVAnd interpolation on anomalies... (This may take some time)
┌ Warning: Preconditioned conjugate gradients method did not converge
└ @ DIVAnd /opt/julia-depot/packages/DIVAnd/4UymR/src/DIVAnd_solve.jl:83


 26.556584 seconds (20.37 M allocations: 6.219 GiB, 13.16% gc time, 77.00% compilation time)


[ Info: Interpolation of anomalies complete.
[ Info: Reconstructed the full interpolated field.
[ Info: Saving the interpolated field...
[ Info: Original Y-axis was descending (north-up). Re-orienting interpolated data.
[ Info: Saving filled raster to: data/filled/woa23_all_p00_01.nc_p_sd.tif.filled.2D.len10.e10.tif
[ Info: Processing finished successfully.
